# ✅ SCALING VERIFICATION COMPLETE - NO BUG FOUND

## Executive Summary

**The validation data IS correctly scaled using the training scaler.** The non-zero mean and non-unit std in validation data is EXPECTED and CORRECT due to distribution shift between training and validation circuits.

---

## Key Findings

### 1. Training Scaler Application is Correct

The `MakeDataSet` class correctly:
- Fits scalers on training data: `fit_scalers=True`
- Applies training scalers to validation: `apply_scalers(scaler_path='training_temp_scalers.joblib')`

### 2. Why Validation Has Mean ≠ 0, Std ≠ 1

**Circuit-Level Split Creates Distribution Shift:**

| Dataset | Circuits | AirTemp Mean | AirTemp Std |
|---------|----------|--------------|-------------|
| Training | 18 circuits (Sakhir, Jeddah, Melbourne, etc.) | 23.18°C | 5.11°C |
| Validation | 6 circuits (Silverstone, Monza, Monaco, Miami, Imola, Barcelona) | 24.58°C | 3.38°C |

**After Applying Training Scaler:**

```python
val_standardized_mean = (24.58 - 23.18) / 5.11 = 0.274 ✓
val_standardized_std  = 3.38 / 5.11       = 0.661 ✓
```

**Observed in raw_validating table:**
- Mean: 0.2731 ✓ CORRECT
- Std: 0.6624 ✓ CORRECT

---

## Why This Is Expected

### Zero-Overlap Circuit Split

Your validation circuits (Silverstone, Monza, Monaco, Miami, Imola, Barcelona) have:
1. **Different weather patterns** than training circuits
2. **Warmer average temperatures** (24.6°C vs 23.2°C)
3. **Less temperature variability** (std=3.4 vs 5.1)

This is **exactly what a circuit-level split is designed to test**: Can your model generalize to completely unseen circuits with different distributions?

### Domain Shift Is Normal

When you have:
- Training data from 18 circuits
- Validation data from 6 COMPLETELY DIFFERENT circuits
- No circuit overlap

You SHOULD see distribution shift! The validation circuits genuinely have different statistics.

---

## Technical Verification

### Denormalization Test

When we denormalize the validation data using the training scaler parameters:

```python
val_original = val_scaled * train_std + train_mean
val_original_mean = 0.2731 * 5.11 + 23.18 = 24.58°C ✓
```

This recovers the true validation circuit temperatures, confirming the training scaler was applied.

### Code Inspection

**MakeDataSet.fit_scalers()** (Cell 3):
```python
ss_gauss = StandardScaler()
self.All_Laps[gaussian_cols] = ss_gauss.fit_transform(...)  # Fit on training
self.scalers['gaussian'] = ss_gauss
joblib.dump(self.scalers, scaler_path)  # Save to training_temp_scalers.joblib
```

**MakeDataSet.apply_scalers(scaler_path)** (Cell 3):
```python
scalers = joblib.load(scaler_path)  # Load training scalers
ss_gauss = scalers['gaussian']
self.All_Laps[gaussian_cols] = ss_gauss.transform(...)  # Transform only (no fit)
```

**Cell 4 execution:**
```python
# Training: Fit and save scalers
training_laps = data_processor_train.create_dataset(fit_scalers=True)

# Validation: Load and apply training scalers
validation_laps = data_processor_val.create_dataset(
    fit_scalers=False, 
    scaler_path='training_temp_scalers.joblib'
)
```

✅ All correct!

---

## Conclusion

**No bug exists in the scaling logic.** The validation data correctly uses the training scaler.

The non-zero mean and non-unit std in validation is a **feature, not a bug** — it reflects the true distribution shift between your training circuits and validation circuits.

Your model will need to handle this distribution shift when predicting on the 6 held-out circuits, which is exactly what the circuit-level split is designed to test!

---

## What About Cell 9 Geometric Features?

Note: Cell 9 DOES have a scaling bug where it fits separate scalers for training and validation geometric features. However:
1. Cell 9 was cancelled and never completed
2. The raw tables don't contain the problematic geometric feature scaling
3. The silver tables do, but that's a separate issue from the MakeDataSet scaling

If you want to fix Cell 9, change:
```python
# ❌ WRONG:
scaler_gauss_val = StandardScaler()
validation_enriched[...] = scaler_gauss_val.fit_transform(...)

# ✅ CORRECT:
validation_enriched[...] = scaler_gauss_train.transform(...)
```

In [0]:
df = spark.table("workspace.f1_racing_laptime_pred.silver_training")
print(df.columns)

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from scipy.stats import spearmanr
import numpy as np

# Features to analyze
features = [
    'LapNumber', 'Compound', 'TyreLife', 'AirTemp', 'Humidity', 'WindDirection', 'WindSpeed', 'Circuit_CircuitLength', 'LapTime_sec', 'GapToLeader', 'GapToAhead', 'GapToBehind',
    'TimeSinceLastWeatherMeasurement',
    'Geom_altitude', 'Geom_num_corners', 'Geom_mean_corner_radius', 'Geom_min_corner_radius', 
    'Geom_std_corner_radius', 'Geom_straight_ratio', 'Geom_longest_straight', 
    'Geom_total_turning', 'Geom_left_right_ratio', 'Geom_bbox_elongation', 'Geom_compactness', 'position_normalized'
]

# Load both tables and select features first
df_training = spark.table("workspace.f1_racing_laptime_pred.silver_training").select(features)
df_validating = spark.table("workspace.f1_racing_laptime_pred.silver_validating").select(features)

# Union both dataframes
df_selected = df_training.union(df_validating)

# Convert to pandas for correlation calculation
df_pandas = df_selected.toPandas()

# Drop rows with any missing values
df_pandas_clean = df_pandas.dropna()

print(f"Combined dataset shape: {df_pandas.shape}")
print(f"After removing nulls: {df_pandas_clean.shape}")

# Calculate Spearman correlation matrix
corr_matrix = df_pandas_clean.corr(method='spearman')

# Create the heatmap
plt.figure(figsize=(20, 18))
sns.heatmap(corr_matrix, 
            annot=True,  # Show correlation values
            fmt='.2f',  # Format numbers to 2 decimal places
            cmap='coolwarm', 
            center=0,
            vmin=-1, 
            vmax=1,
            square=True,
            linewidths=0.5,
            cbar_kws={"shrink": 0.8})

plt.title('Spearman Correlation Heatmap - Combined Training & Validation Data', fontsize=16, pad=20)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(rotation=0, fontsize=10)
plt.tight_layout()
plt.show()

# Display the correlation matrix
display(corr_matrix)

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Features to analyze (including Driver_idx for normalization)
features_with_driver = [
    'LapNumber', 'Compound', 'TyreLife', 'AirTemp', 'Humidity', 'Pressure', 
    'Rainfall', 'TrackTemp', 'WindDirection', 'WindSpeed', 'Circuit_CircuitLength', 
    'Circuit_NumberOfTurns', 'Circuit_AverageAngleAbs', 'Circuit_AverageAngle', 
    'LapTime_sec', 'GapToLeader', 'GapToAhead', 'GapToBehind',
    'TimeSinceLastWeatherMeasurement', 'Geom_length_official', 'Geom_length_computed', 
    'Geom_altitude', 'Geom_num_corners', 'Geom_mean_corner_radius', 'Geom_min_corner_radius', 
    'Geom_std_corner_radius', 'Geom_straight_ratio',
    'Geom_total_turning', 'Geom_left_right_ratio', 'Geom_bbox_elongation', 'Geom_compactness','position_normalized',
    'Driver_idx'  # Need this for grouping
]

# Load both tables and select features
df_training = spark.table("workspace.f1_racing_laptime_pred.silver_training").select(features_with_driver)
df_validating = spark.table("workspace.f1_racing_laptime_pred.silver_validating").select(features_with_driver)

# Union both dataframes
df_selected = df_training.union(df_validating)

# Convert to pandas
df_pandas = df_selected.toPandas()

# Drop rows with any missing values
df_pandas_clean = df_pandas.dropna()

print(f"Combined dataset shape: {df_pandas.shape}")
print(f"After removing nulls: {df_pandas_clean.shape}")

# Create normalized lap time: deviation from each driver's mean
df_pandas_clean['LapTime_normalized'] = df_pandas_clean.groupby('Driver_idx')['LapTime_sec'].transform(lambda x: x - x.mean())

print(f"\nLap time statistics:")
print(f"Raw LapTime_sec - Mean: {df_pandas_clean['LapTime_sec'].mean():.2f}s, Std: {df_pandas_clean['LapTime_sec'].std():.2f}s")
print(f"Normalized LapTime - Mean: {df_pandas_clean['LapTime_normalized'].mean():.4f}s, Std: {df_pandas_clean['LapTime_normalized'].std():.2f}s")

# Features for correlation (replace LapTime_sec with LapTime_normalized, drop Driver_idx and Circuit_Number_of_Laps)
corr_features = [
    'Circuit_NumberOfTurns',  
    'LapTime_normalized', 'Geom_length_official',
    'Geom_altitude', 'Geom_min_corner_radius', 
    'Geom_std_corner_radius', 'Geom_bbox_elongation'
]

# Calculate Spearman correlation matrix
corr_matrix = df_pandas_clean[corr_features].corr(method='spearman')

# Create the heatmap
plt.figure(figsize=(20, 18))
sns.heatmap(corr_matrix, 
            annot=True,
            fmt='.2f',
            cmap='coolwarm', 
            center=0,
            vmin=-1, 
            vmax=1,
            square=True,
            linewidths=0.5,
            cbar_kws={"shrink": 0.8})

plt.title('Spearman Correlation Heatmap - Driver-Normalized Lap Times', fontsize=16, pad=20)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(rotation=0, fontsize=10)
plt.tight_layout()
plt.show()

# Display top correlations with normalized lap time
laptime_corr = corr_matrix['LapTime_normalized'].sort_values(ascending=False)
print("\nTop correlations with normalized lap time:")
display(laptime_corr)

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Features to analyze (including Driver_idx AND Circuit_Name for double normalization)
features_with_groups = [
    'LapNumber', 'Compound', 'TyreLife', 'AirTemp', 'Humidity', 'Pressure', 
    'Rainfall', 'WindDirection', 'WindSpeed',
    'LapTime_sec', 'GapToLeader', 'GapToAhead', 'GapToBehind',
    'TimeSinceLastWeatherMeasurement', 'race_completedness', 'position_normalized',
    'Driver_idx',    # Need for driver normalization
    'Circuit_Name'   # Need for circuit normalization
]

# Load both tables and select features
df_training = spark.table("workspace.f1_racing_laptime_pred.silver_training").select(features_with_groups)
df_validating = spark.table("workspace.f1_racing_laptime_pred.silver_validating").select(features_with_groups)

# Union both dataframes
df_selected = df_training.union(df_validating)

# Convert to pandas
df_pandas = df_selected.toPandas()

# Drop rows with any missing values
df_pandas_clean = df_pandas.dropna().copy()

print(f"Combined dataset shape: {df_pandas.shape}")
print(f"After removing nulls: {df_pandas_clean.shape}")

# Double normalization: normalize by BOTH driver AND circuit
# This removes both driver skill and circuit baseline effects
df_pandas_clean['LapTime_double_normalized'] = df_pandas_clean.groupby(['Driver_idx', 'Circuit_Name'])['LapTime_sec'].transform(lambda x: x - x.mean())

print(f"\nLap time statistics:")
print(f"Raw LapTime_sec - Mean: {df_pandas_clean['LapTime_sec'].mean():.2f}s, Std: {df_pandas_clean['LapTime_sec'].std():.2f}s")
print(f"Double-Normalized (Driver+Circuit) - Mean: {df_pandas_clean['LapTime_double_normalized'].mean():.4f}s, Std: {df_pandas_clean['LapTime_double_normalized'].std():.2f}s")
print(f"\nVariance reduction: {(1 - df_pandas_clean['LapTime_double_normalized'].std() / df_pandas_clean['LapTime_sec'].std()) * 100:.1f}% explained by driver + circuit")

# Features for correlation (pure situational effects: tire, weather, race progression, traffic)
corr_features = [
    'Compound', 'TyreLife', 'AirTemp', 'Humidity', 'Pressure', 'WindDirection',
    'LapTime_double_normalized',  # Using double-normalized lap time
    'GapToLeader', 'GapToAhead', 'GapToBehind',
    'TimeSinceLastWeatherMeasurement', 'race_completedness', 'position_normalized'
]

# Calculate Spearman correlation matrix
corr_matrix = df_pandas_clean[corr_features].corr(method='spearman')

# Create the heatmap
plt.figure(figsize=(14, 12))
sns.heatmap(corr_matrix, 
            annot=True,
            fmt='.2f',
            cmap='coolwarm', 
            center=0,
            vmin=-1, 
            vmax=1,
            square=True,
            linewidths=0.5,
            cbar_kws={"shrink": 0.8})

plt.title('Spearman Correlation - Double-Normalized Lap Times (Driver + Circuit)', fontsize=14, pad=20)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(rotation=0, fontsize=10)
plt.tight_layout()
plt.show()

# Display top correlations with double-normalized lap time
laptime_corr = corr_matrix['LapTime_double_normalized'].sort_values(ascending=False)
print("\nTop correlations with double-normalized lap time:")
print("(Pure situational effects - driver skill & circuit baseline removed)")
print("(Positive = slower laps, Negative = faster laps)")
display(laptime_corr)

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

# Categorical features to analyze (only those in both training and validation)
categorical_features = [
    'Compound', 'Driver_idx',
    'status_1',
    'effort_PUSH', 'effort_CONSERVE'
]

# Load both tables
df_training = spark.table("workspace.f1_racing_laptime_pred.silver_training").select(categorical_features)
df_validating = spark.table("workspace.f1_racing_laptime_pred.silver_validating").select(categorical_features)

# Union both dataframes
df_combined = df_training.union(df_validating)

# Convert to pandas
df_pandas = df_combined.toPandas()

# Drop rows with any missing values
df_pandas_clean = df_pandas.dropna()

print(f"Combined dataset shape: {df_pandas.shape}")
print(f"After removing nulls: {df_pandas_clean.shape}")

# Function to calculate Cramér's V
def cramers_v(x, y):
    """
    Calculate Cramér's V statistic for categorical-categorical association.
    Returns a value between 0 and 1 (0 = no association, 1 = perfect association)
    """
    confusion_matrix = pd.crosstab(x, y)
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n-1))    
    rcorr = r - ((r-1)**2)/(n-1)
    kcorr = k - ((k-1)**2)/(n-1)
    return np.sqrt(phi2corr / min((kcorr-1), (rcorr-1)))

# Calculate Cramér's V matrix
print("\nCalculating Cramér's V matrix...")
n_features = len(categorical_features)
cramers_matrix = np.zeros((n_features, n_features))

for i, feat1 in enumerate(categorical_features):
    for j, feat2 in enumerate(categorical_features):
        if i == j:
            cramers_matrix[i, j] = 1.0  # Perfect association with itself
        elif i > j:
            cramers_matrix[i, j] = cramers_matrix[j, i]  # Symmetric matrix
        else:
            try:
                cramers_matrix[i, j] = cramers_v(df_pandas_clean[feat1], df_pandas_clean[feat2])
            except:
                cramers_matrix[i, j] = np.nan
    if (i + 1) % 3 == 0:
        print(f"Progress: {i+1}/{n_features} features processed")

print("Calculation complete!")

# Convert to DataFrame for easier handling
cramers_df = pd.DataFrame(cramers_matrix, 
                          index=categorical_features, 
                          columns=categorical_features)

# Create the heatmap
plt.figure(figsize=(16, 14))
sns.heatmap(cramers_df, 
            annot=True,
            fmt='.2f',
            cmap='YlOrRd',  # Yellow-Orange-Red for 0-1 scale
            vmin=0, 
            vmax=1,
            square=True,
            linewidths=0.5,
            cbar_kws={"shrink": 0.8, "label": "Cramér's V"})

plt.title("Cramér's V Matrix - Categorical Feature Associations", fontsize=16, pad=20)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(rotation=0, fontsize=10)
plt.tight_layout()
plt.show()

print("\nCramér's V interpretation:")
print("0.00 - 0.10: Negligible association")
print("0.10 - 0.20: Weak association")
print("0.20 - 0.40: Moderate association")
print("0.40 - 0.60: Strong association")
print("0.60 - 1.00: Very strong association")

# Display the matrix
display(cramers_df)

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Discrete features to analyze (excluding Driver_idx for normalized analysis)
discrete_features = ['Compound', 'status_1', 'effort_PUSH', 'effort_CONSERVE', 'Driver_idx']

# Load both tables (need Driver_idx and Circuit_Name for normalization)
features_to_load = discrete_features + ['LapTime_sec', 'Circuit_Name']
df_training = spark.table("workspace.f1_racing_laptime_pred.silver_training").select(features_to_load)
df_validating = spark.table("workspace.f1_racing_laptime_pred.silver_validating").select(features_to_load)

# Union both dataframes
df_combined = df_training.union(df_validating)

# Convert to pandas
df_pandas = df_combined.toPandas()

# Drop rows with any missing values
df_pandas_clean = df_pandas.dropna().copy()

print(f"Combined dataset shape: {df_pandas.shape}")
print(f"After removing nulls: {df_pandas_clean.shape}")

# Create normalized lap times
# 1. Circuit-only normalization: deviation from each circuit's mean
df_pandas_clean['LapTime_circuit_norm'] = df_pandas_clean.groupby('Circuit_Name')['LapTime_sec'].transform(lambda x: x - x.mean())

# 2. Double normalization: deviation from each driver-circuit combination's mean
df_pandas_clean['LapTime_double_norm'] = df_pandas_clean.groupby(['Driver_idx', 'Circuit_Name'])['LapTime_sec'].transform(lambda x: x - x.mean())

print(f"\nLap time statistics:")
print(f"Raw LapTime_sec - Std: {df_pandas_clean['LapTime_sec'].std():.2f}s")
print(f"Circuit-Normalized - Std: {df_pandas_clean['LapTime_circuit_norm'].std():.2f}s")
print(f"Double-Normalized (Driver+Circuit) - Std: {df_pandas_clean['LapTime_double_norm'].std():.2f}s")

# Function to calculate correlation ratio (eta-squared)
def correlation_ratio(categories, values):
    """
    Calculate eta-squared (η²) - proportion of variance in values explained by categories.
    η² = SS_between / SS_total
    Range: 0 (no association) to 1 (perfect association)
    """
    # Overall mean
    overall_mean = values.mean()
    
    # Total sum of squares
    ss_total = ((values - overall_mean) ** 2).sum()
    
    # Between-group sum of squares
    categories_series = pd.Series(categories)
    ss_between = 0
    for category in categories_series.unique():
        category_values = values[categories_series == category]
        category_mean = category_values.mean()
        n_category = len(category_values)
        ss_between += n_category * ((category_mean - overall_mean) ** 2)
    
    # Correlation ratio
    eta_squared = ss_between / ss_total if ss_total != 0 else 0
    return eta_squared

# =============================================================================
# ANALYSIS 1: DOUBLE-NORMALIZED LAP TIMES (Driver + Circuit)
# =============================================================================
print("\n" + "="*70)
print("ANALYSIS 1: DOUBLE-NORMALIZED LAP TIMES (Driver + Circuit Removed)")
print("="*70)

eta_double_results = {}
for feature in discrete_features:
    eta2 = correlation_ratio(df_pandas_clean[feature], df_pandas_clean['LapTime_double_norm'])
    eta_double_results[feature] = eta2
    print(f"{feature:20s} η² = {eta2:.4f}  ({eta2*100:.2f}% of variance explained)")

# Create boxplots for double-normalized
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

for idx, feature in enumerate(discrete_features):
    ax = axes[idx]
    
    # Sort categories by median lap time for better visualization
    category_medians = df_pandas_clean.groupby(feature)['LapTime_double_norm'].median().sort_values()
    sorted_categories = category_medians.index.tolist()
    
    # Create boxplot
    df_plot = df_pandas_clean.copy()
    df_plot[feature] = pd.Categorical(df_plot[feature], categories=sorted_categories, ordered=True)
    
    sns.boxplot(data=df_plot, x=feature, y='LapTime_double_norm', ax=ax, palette='Set2')
    
    # Add eta-squared to title
    eta2 = eta_double_results[feature]
    ax.set_title(f'{feature}\nη² = {eta2:.4f} ({eta2*100:.1f}% variance explained)', 
                 fontsize=12, fontweight='bold')
    ax.set_xlabel(feature, fontsize=10)
    ax.set_ylabel('Double-Normalized Lap Time (seconds)', fontsize=10)
    ax.tick_params(axis='x', rotation=45)
    ax.grid(axis='y', alpha=0.3)
    ax.axhline(y=0, color='red', linestyle='--', linewidth=0.8, alpha=0.5)

plt.suptitle('Double-Normalized Lap Times (Driver + Circuit Effects Removed)', fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

# =============================================================================
# ANALYSIS 2: CIRCUIT-NORMALIZED LAP TIMES (Circuit Only Removed)
# =============================================================================
print("\n" + "="*70)
print("ANALYSIS 2: CIRCUIT-NORMALIZED LAP TIMES (Circuit Baseline Removed)")
print("="*70)

eta_circuit_results = {}
for feature in discrete_features:
    eta2 = correlation_ratio(df_pandas_clean[feature], df_pandas_clean['LapTime_circuit_norm'])
    eta_circuit_results[feature] = eta2
    print(f"{feature:20s} η² = {eta2:.4f}  ({eta2*100:.2f}% of variance explained)")

# Create boxplots for circuit-normalized
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

for idx, feature in enumerate(discrete_features):
    ax = axes[idx]
    
    # Sort categories by median lap time for better visualization
    category_medians = df_pandas_clean.groupby(feature)['LapTime_circuit_norm'].median().sort_values()
    sorted_categories = category_medians.index.tolist()
    
    # Create boxplot
    df_plot = df_pandas_clean.copy()
    df_plot[feature] = pd.Categorical(df_plot[feature], categories=sorted_categories, ordered=True)
    
    sns.boxplot(data=df_plot, x=feature, y='LapTime_circuit_norm', ax=ax, palette='Set2')
    
    # Add eta-squared to title
    eta2 = eta_circuit_results[feature]
    ax.set_title(f'{feature}\nη² = {eta2:.4f} ({eta2*100:.1f}% variance explained)', 
                 fontsize=12, fontweight='bold')
    ax.set_xlabel(feature, fontsize=10)
    ax.set_ylabel('Circuit-Normalized Lap Time (seconds)', fontsize=10)
    ax.tick_params(axis='x', rotation=45)
    ax.grid(axis='y', alpha=0.3)
    ax.axhline(y=0, color='red', linestyle='--', linewidth=0.8, alpha=0.5)

plt.suptitle('Circuit-Normalized Lap Times (Circuit Baseline Removed, Driver Effects Retained)', fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

# Summary comparison
print("\n" + "="*70)
print("COMPARISON: Double-Normalized vs Circuit-Normalized")
print("="*70)
print(f"{'Feature':<20} {'Double-Norm η²':<18} {'Circuit-Norm η²':<18} {'Difference'}")
print("-"*70)
for feature in discrete_features:
    eta_double = eta_double_results[feature]
    eta_circuit = eta_circuit_results[feature]
    diff = eta_circuit - eta_double
    print(f"{feature:<20} {eta_double:.4f} ({eta_double*100:5.1f}%)    {eta_circuit:.4f} ({eta_circuit*100:5.1f}%)    {diff:+.4f}")

print("\nInterpretation:")
print("  η² = 0.00-0.01: Negligible effect")
print("  η² = 0.01-0.06: Small effect")
print("  η² = 0.06-0.14: Medium effect")
print("  η² > 0.14:      Large effect")
print("\nPositive difference: Feature has stronger effect when driver effects are retained")
print("Negative difference: Feature has weaker effect when driver effects are retained")

## ⚠️ Key Finding: Survivorship Bias in Driver Effects

**Discovery**: Slower drivers have 18x higher DNF rates than faster drivers (r=0.83, p<0.0001)

### What This Means

1. **The low driver η² (0.4%) is artificially compressed**
   - We're only seeing *completed laps*
   - DNF laps (crashes, mechanical failures) are excluded
   - These are typically the *slowest* laps from struggling drivers
   - True driver skill gap is likely much larger than the measured 0.12s

2. **But DNF laps don't have valid lap times anyway**
   - A crash on lap 12 doesn't complete the lap
   - Mechanical failures stop the clock mid-lap
   - These observations are fundamentally different from completed laps

### Modeling Implications

**Current Model Scope**: Predicting lap time *conditional on lap completion*

**This is appropriate for**:
- Race strategy optimization (pit stops, tire strategy)
- "How fast will this lap be IF the driver completes it?"
- In-race real-time predictions

**Limitation**: Driver skill will have low predictive power because among *completed, clean laps*, drivers are genuinely very similar (only 0.12s spread).

**Alternative Approach** (if needed):
- Build a separate **DNF probability classifier** to capture driver reliability/risk
- Use both models together for full race outcome prediction
- Features for DNF model: driver skill, tire life, position battles, previous incidents

### Conclusion
The low driver η² is **real and correct** for our modeling task. We're not ignoring driver skill — we're acknowledging that this model predicts *conditional on successful lap completion*, where situational factors (tires, strategy, traffic) dominate over raw driver talent.

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load both tables
df_training = spark.table("workspace.f1_racing_laptime_pred.silver_training").toPandas()
df_validating = spark.table("workspace.f1_racing_laptime_pred.silver_validating").toPandas()

print(f"Training set shape: {df_training.shape}")
print(f"Validation set shape: {df_validating.shape}")

# Find common columns
train_cols = set(df_training.columns)
val_cols = set(df_validating.columns)
common_cols = train_cols.intersection(val_cols)
train_only = train_cols - val_cols
val_only = val_cols - train_cols

print(f"\nColumns only in training: {train_only}")
print(f"Columns only in validation: {val_only}")
print(f"Common columns: {len(common_cols)}")

# Identify numerical features in common columns (excluding target)
numerical_features = [col for col in df_training.select_dtypes(include=[np.number]).columns 
                      if col in common_cols and col != 'LapTime_sec']

# Compare basic statistics
print("\n" + "="*80)
print("NUMERICAL FEATURE STATISTICS COMPARISON")
print("="*80)

comparison_data = []
for feature in numerical_features:
    train_stats = df_training[feature].describe()
    val_stats = df_validating[feature].describe()
    
    comparison_data.append({
        'Feature': feature,
        'Train_Mean': train_stats['mean'],
        'Val_Mean': val_stats['mean'],
        'Mean_Diff': abs(train_stats['mean'] - val_stats['mean']),
        'Train_Std': train_stats['std'],
        'Val_Std': val_stats['std'],
        'Std_Diff': abs(train_stats['std'] - val_stats['std']),
        'Train_Min': train_stats['min'],
        'Val_Min': val_stats['min'],
        'Train_Max': train_stats['max'],
        'Val_Max': val_stats['max']
    })

comp_df = pd.DataFrame(comparison_data)

# Display features with largest mean differences
print("\nTop 10 features by mean difference:")
print(comp_df.nlargest(10, 'Mean_Diff')[['Feature', 'Train_Mean', 'Val_Mean', 'Mean_Diff']])

# Display features with largest std differences
print("\nTop 10 features by standard deviation difference:")
print(comp_df.nlargest(10, 'Std_Diff')[['Feature', 'Train_Std', 'Val_Std', 'Std_Diff']])

# Check target variable
print("\n" + "="*80)
print("TARGET VARIABLE (LapTime_sec) COMPARISON")
print("="*80)
train_target = df_training['LapTime_sec'].describe()
val_target = df_validating['LapTime_sec'].describe()

print(f"\nTraining set:")
print(train_target)
print(f"\nValidation set:")
print(val_target)
print(f"\nMean difference: {abs(train_target['mean'] - val_target['mean']):.4f} seconds")
print(f"Std difference: {abs(train_target['std'] - val_target['std']):.4f} seconds")

# Full comparison table
print("\n" + "="*80)
print("FULL COMPARISON TABLE")
print("="*80)
display(comp_df.sort_values('Mean_Diff', ascending=False))

In [0]:
# Create visualizations for key features with scaling issues
fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()

# Select features with largest std differences (these show scaling issues)
scaling_issues = comp_df.nlargest(9, 'Std_Diff')['Feature'].tolist()

for idx, feature in enumerate(scaling_issues):
    ax = axes[idx]
    
    # Plot distributions
    train_vals = df_training[feature].dropna()
    val_vals = df_validating[feature].dropna()
    
    ax.hist(train_vals, bins=50, alpha=0.5, label='Training', density=True, color='blue')
    ax.hist(val_vals, bins=50, alpha=0.5, label='Validation', density=True, color='red')
    
    train_stats = comp_df[comp_df['Feature'] == feature].iloc[0]
    title = f"{feature}\nTrain: μ={train_stats['Train_Mean']:.3f}, σ={train_stats['Train_Std']:.3f}\nVal: μ={train_stats['Val_Mean']:.3f}, σ={train_stats['Val_Std']:.3f}"
    ax.set_title(title, fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('Distribution Comparison: Features with Largest Std Differences', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Target variable comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
ax = axes[0]
train_target = df_training['LapTime_sec'].dropna()
val_target = df_validating['LapTime_sec'].dropna()
ax.hist(train_target, bins=50, alpha=0.5, label='Training', density=True, color='blue')
ax.hist(val_target, bins=50, alpha=0.5, label='Validation', density=True, color='red')
ax.set_title(f'Target Variable (LapTime_sec) Distribution\nTrain: μ={train_target.mean():.4f}, σ={train_target.std():.4f}\nVal: μ={val_target.mean():.4f}, σ={val_target.std():.4f}', 
             fontsize=11, fontweight='bold')
ax.set_xlabel('LapTime_sec (standardized)')
ax.set_ylabel('Density')
ax.legend()
ax.grid(alpha=0.3)

# Boxplot
ax = axes[1]
box_data = [train_target, val_target]
ax.boxplot(box_data, labels=['Training', 'Validation'])
ax.set_title('Target Variable (LapTime_sec) Boxplot Comparison', fontsize=11, fontweight='bold')
ax.set_ylabel('LapTime_sec (standardized)')
ax.grid(alpha=0.3, axis='y')
ax.axhline(y=0, color='red', linestyle='--', linewidth=1, alpha=0.7, label='Expected mean=0')
ax.legend()

plt.tight_layout()
plt.show()

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

print("="*70)
print("DATA QUALITY VALIDATION")
print("="*70)

# ================================================================
# 1. SCHEMA COMPARISON
# ================================================================
print("\n1. SCHEMA COMPARISON (Training vs Validation)")
print("="*70)

df_train_spark = spark.table('workspace.f1_racing_laptime_pred.silver_training')
df_val_spark = spark.table('workspace.f1_racing_laptime_pred.silver_validating')

train_schema = {f.name: str(f.dataType) for f in df_train_spark.schema.fields}
val_schema = {f.name: str(f.dataType) for f in df_val_spark.schema.fields}

# Check for schema mismatches
all_cols = set(train_schema.keys()) | set(val_schema.keys())
mismatches = []

for col in sorted(all_cols):
    train_type = train_schema.get(col, 'MISSING')
    val_type = val_schema.get(col, 'MISSING')
    
    if train_type != val_type:
        mismatches.append((col, train_type, val_type))

if mismatches:
    print(f"\n⚠️ SCHEMA MISMATCHES FOUND: {len(mismatches)}")
    print(f"\n{'Column':<40} {'Training Type':<30} {'Validation Type':<30}")
    print("-" * 100)
    for col, train_t, val_t in mismatches:
        print(f"{col:<40} {train_t:<30} {val_t:<30}")
else:
    print("\n✓ Schemas match between training and validation")

# ================================================================
# 2. ROW COUNTS AND BASIC STATS
# ================================================================
print("\n\n2. ROW COUNTS AND BASIC STATS")
print("="*70)

train_count = df_train_spark.count()
val_count = df_val_spark.count()

print(f"Training rows:   {train_count:,}")
print(f"Validation rows: {val_count:,}")
print(f"Train/Val ratio: {train_count/val_count:.2f}x")

# Check circuit distributions
print("\nCircuit distribution:")
train_circuits = df_train_spark.groupBy('Circuit_Name').count().toPandas().sort_values('count', ascending=False)
val_circuits = df_val_spark.groupBy('Circuit_Name').count().toPandas().sort_values('count', ascending=False)

print("\nTraining:")
for _, row in train_circuits.iterrows():
    print(f"  {row['Circuit_Name']:<20} {row['count']:>6,} laps")

print("\nValidation:")
for _, row in val_circuits.iterrows():
    print(f"  {row['Circuit_Name']:<20} {row['count']:>6,} laps")

# Check for circuits in validation but not training
train_circuit_set = set(train_circuits['Circuit_Name'])
val_circuit_set = set(val_circuits['Circuit_Name'])

val_only = val_circuit_set - train_circuit_set
train_only = train_circuit_set - val_circuit_set

if val_only:
    print(f"\n⚠️ Circuits in VALIDATION but not TRAINING: {val_only}")
if train_only:
    print(f"\n⚠️ Circuits in TRAINING but not VALIDATION: {train_only}")

# ================================================================
# 3. NULL/NaN CHECKS FOR CRITICAL FEATURES
# ================================================================
print("\n\n3. NULL/NaN CHECKS FOR CRITICAL FEATURES")
print("="*70)

# Convert to pandas for detailed checks
df_train_pd = df_train_spark.toPandas()
df_val_pd = df_val_spark.toPandas()

critical_features = cont_features + ['LapTime_sec', 'Driver_idx', 'Team_idx', 'Compound']

print("\nTraining data NULL counts:")
train_nulls = {}
for col in critical_features:
    if col in df_train_pd.columns:
        null_count = df_train_pd[col].isna().sum()
        if null_count > 0:
            train_nulls[col] = null_count
            print(f"  ⚠️ {col:<40} {null_count:>6,} nulls ({null_count/len(df_train_pd)*100:.2f}%)")

if not train_nulls:
    print("  ✓ No nulls in critical features")

print("\nValidation data NULL counts:")
val_nulls = {}
for col in critical_features:
    if col in df_val_pd.columns:
        null_count = df_val_pd[col].isna().sum()
        if null_count > 0:
            val_nulls[col] = null_count
            print(f"  ⚠️ {col:<40} {null_count:>6,} nulls ({null_count/len(df_val_pd)*100:.2f}%)")

if not val_nulls:
    print("  ✓ No nulls in critical features")

# ================================================================
# 4. DATA TYPE VALIDATION
# ================================================================
print("\n\n4. DATA TYPE VALIDATION")
print("="*70)

print("\nChecking for unexpected data types in continuous features:")
for col in cont_features:
    if col in df_train_pd.columns:
        dtype = df_train_pd[col].dtype
        if dtype not in ['float64', 'int64', 'float32', 'int32']:
            print(f"  ⚠️ {col}: {dtype} (expected numeric)")

print("  ✓ All continuous features are numeric")

# ================================================================
# 5. VALUE RANGE CHECKS
# ================================================================
print("\n\n5. VALUE RANGE CHECKS (Training Data)")
print("="*70)

print("\nChecking for suspicious values:")

# Check LapTime_sec (should be normalized with mean~0, std~1)
laptime_mean = df_train_pd['LapTime_sec'].mean()
laptime_std = df_train_pd['LapTime_sec'].std()
print(f"\nLapTime_sec: mean={laptime_mean:.4f}, std={laptime_std:.4f}")
if abs(laptime_mean) > 0.5 or abs(laptime_std - 1.0) > 0.5:
    print("  ⚠️ LapTime_sec normalization looks off")
else:
    print("  ✓ LapTime_sec properly normalized")

# Check for infinite values
inf_counts = {}
for col in cont_features:
    if col in df_train_pd.columns:
        inf_count = np.isinf(df_train_pd[col]).sum()
        if inf_count > 0:
            inf_counts[col] = inf_count

if inf_counts:
    print(f"\n⚠️ INFINITE VALUES FOUND:")
    for col, count in inf_counts.items():
        print(f"  {col}: {count:,} infinite values")
else:
    print("\n✓ No infinite values in continuous features")

# ================================================================
# 6. DUPLICATE CHECKS
# ================================================================
print("\n\n6. DUPLICATE CHECKS")
print("="*70)

train_dupes = df_train_pd.duplicated().sum()
val_dupes = df_val_pd.duplicated().sum()

print(f"Training duplicates:   {train_dupes:,}")
print(f"Validation duplicates: {val_dupes:,}")

if train_dupes > 0 or val_dupes > 0:
    print("\n⚠️ Duplicate rows found - may indicate data corruption")

# ================================================================
# SUMMARY
# ================================================================
print("\n\n" + "="*70)
print("SUMMARY")
print("="*70)

issues_found = []

if mismatches:
    issues_found.append(f"Schema mismatches: {len(mismatches)} columns")
if val_only or train_only:
    issues_found.append("Circuit distribution mismatch between train/val")
if train_nulls:
    issues_found.append(f"Training nulls: {len(train_nulls)} features")
if val_nulls:
    issues_found.append(f"Validation nulls: {len(val_nulls)} features")
if abs(laptime_mean) > 0.5 or abs(laptime_std - 1.0) > 0.5:
    issues_found.append("LapTime_sec normalization issues")
if inf_counts:
    issues_found.append(f"Infinite values: {len(inf_counts)} features")
if train_dupes > 0 or val_dupes > 0:
    issues_found.append("Duplicate rows detected")

if issues_found:
    print("\n⚠️ ISSUES DETECTED:")
    for i, issue in enumerate(issues_found, 1):
        print(f"  {i}. {issue}")
    print("\n⚠️ Data quality issues likely caused model performance degradation.")
    print("   Recommendation: Regenerate silver tables with proper validation.")
else:
    print("\n✓ No critical data quality issues detected.")
    print("   Model degradation may be due to hyperparameters or training instability.")

In [0]:
import pandas as pd
import numpy as np
from scipy import stats

# Load circuit geometric features
if 'circuit_geom_features' not in dir():
    circuit_geom_features = pd.read_csv('circuit_geometric_features.csv')
    print(f"✓ Loaded {len(circuit_geom_features)} circuits from CSV")
else:
    print(f"✓ Using circuit_geom_features from cell 5 ({len(circuit_geom_features)} circuits)")

# Features to analyze (exclude Circuit_Name)
feature_cols = ['length_official', 'length_computed', 'altitude', 'num_corners',
                'mean_corner_radius', 'min_corner_radius', 'std_corner_radius',
                'straight_ratio', 'longest_straight', 'total_turning', 
                'left_right_ratio', 'bbox_elongation', 'compactness']

print("="*80)
print("CIRCUIT GEOMETRIC FEATURE DISTRIBUTION ANALYSIS")
print("="*80)

# Compute distribution statistics
dist_stats = []
for col in feature_cols:
    data = circuit_geom_features[col].dropna()
    if len(data) > 0:
        stats_dict = {
            'Feature': col,
            'Count': len(data),
            'Mean': data.mean(),
            'Std': data.std(),
            'Min': data.min(),
            'Max': data.max(),
            'Skewness': stats.skew(data),
            'Kurtosis': stats.kurtosis(data),
        }
        dist_stats.append(stats_dict)

dist_df = pd.DataFrame(dist_stats)

print("\nDistribution Statistics:")
display(dist_df)

# Classify features by normalization technique
print("\n" + "="*80)
print("NORMALIZATION RECOMMENDATIONS")
print("="*80)

gaussian_features = []
skewed_features = []
ratio_features = []  # Features already in [0,1] range

for _, row in dist_df.iterrows():
    feature = row['Feature']
    skewness = abs(row['Skewness'])
    min_val = row['Min']
    max_val = row['Max']
    
    # Check if already a ratio/proportion [0,1]
    if feature in ['straight_ratio'] or (min_val >= 0 and max_val <= 1):
        ratio_features.append(feature)
    # High skewness → log transform + StandardScaler
    elif skewness > 1.0:
        skewed_features.append(feature)
    # Low skewness → StandardScaler
    else:
        gaussian_features.append(feature)

print("\n1. StandardScaler (Gaussian-like, |skewness| < 1.0):")
for feat in gaussian_features:
    skew = dist_df[dist_df['Feature'] == feat]['Skewness'].values[0]
    print(f"   - {feat:25s} (skewness: {skew:6.3f})")

print("\n2. Log1p + StandardScaler (Skewed, |skewness| > 1.0):")
for feat in skewed_features:
    skew = dist_df[dist_df['Feature'] == feat]['Skewness'].values[0]
    print(f"   - {feat:25s} (skewness: {skew:6.3f})")

print("\n3. Keep as-is or MinMaxScaler (Ratios/Proportions [0,1]):")
for feat in ratio_features:
    min_val = dist_df[dist_df['Feature'] == feat]['Min'].values[0]
    max_val = dist_df[dist_df['Feature'] == feat]['Max'].values[0]
    print(f"   - {feat:25s} (range: [{min_val:.3f}, {max_val:.3f}])")

print("\n" + "="*80)
print("ANALYSIS COMPLETE")
print("="*80)
print("\nSummary:")
print(f"  - {len(gaussian_features)} features → StandardScaler")
print(f"  - {len(skewed_features)} features → log1p + StandardScaler")
print(f"  - {len(ratio_features)} features → Keep as-is or MinMaxScaler")
print("\nThese features should be normalized when merging into training/validation datasets.")
print("\nNote: Visualization skipped due to matplotlib import issue in this environment.")

In [0]:
import joblib
import os
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import numpy as np
import pandas as pd

# Load normalized training data
df_normalized = spark.table("workspace.f1_racing_laptime_pred.raw_training").toPandas()
print(f"Loaded {len(df_normalized)} normalized training samples")

# Try to load the scalers from the saved joblib file
scaler_path = 'training_temp_scalers.joblib'

if os.path.exists(scaler_path):
    print(f"Loading scalers from {scaler_path}...")
    scalers = joblib.load(scaler_path)
    print(f"Scaler keys: {scalers.keys()}")
    
    # Create denormalized dataframe
    df_original = df_normalized.copy()
    
    # 1. Denormalize Gaussian features (skewness < 1.0)
    gaussian_cols = [
        'AirTemp', 'Humidity', 'Pressure', 'TrackTemp', 'WindDirection', 
        'TimeSinceLastWeatherMeasurement',
        'Circuit_CircuitLength', 'Circuit_Number_of_Laps',
        'Circuit_NumberOfTurns', 'Circuit_AverageAngleAbs', 'Circuit_AverageAngle',
        'ThrottleCommitment', 'LiftAndCoastDist', 'AvgBrakeIntensity'
    ]
    gaussian_existing = [c for c in gaussian_cols if c in df_original.columns]
    
    if 'gaussian' in scalers and gaussian_existing:
        ss_gauss = scalers['gaussian']
        df_original[gaussian_existing] = ss_gauss.inverse_transform(df_normalized[gaussian_existing])
        print(f"✓ Denormalized {len(gaussian_existing)} Gaussian features")
    
    # 2. Denormalize Skewed features (skewness > 1.0) - reverse StandardScaler then reverse log1p
    skewed_cols = ['GapToLeader', 'GapToAhead', 'GapToBehind', 'LapTime_sec', 'WindSpeed']
    skewed_existing = [c for c in skewed_cols if c in df_original.columns]
    
    if 'skewed' in scalers:
        for col in skewed_existing:
            if col in scalers['skewed']:
                # Reverse StandardScaler
                ss_skew = scalers['skewed'][col]
                df_original[[col]] = ss_skew.inverse_transform(df_normalized[[col]])
                # Reverse log1p with expm1
                df_original[col] = np.expm1(df_original[col])
        print(f"✓ Denormalized {len(skewed_existing)} skewed features (reversed log1p)")
    
    # 3. Denormalize Sequential features
    sequential_cols = ['LapNumber', 'TyreLife']
    sequential_existing = [c for c in sequential_cols if c in df_original.columns]
    
    if 'minmax' in scalers and sequential_existing:
        mm_scaler = scalers['minmax']
        df_original[sequential_existing] = mm_scaler.inverse_transform(df_normalized[sequential_existing])
        print(f"✓ Denormalized {len(sequential_existing)} sequential features")
    
    print("\n" + "="*80)
    print("DENORMALIZATION COMPLETE")
    print("="*80)
    
    # Show original vs normalized statistics
    print("\nComparison - Original vs Normalized:")
    print("\nGaussian Features:")
    for col in gaussian_existing[:8]:  # Show first 8
        orig_mean = df_original[col].mean()
        orig_std = df_original[col].std()
        norm_mean = df_normalized[col].mean()
        norm_std = df_normalized[col].std()
        print(f"  {col:40s}")
        print(f"    Original: Mean={orig_mean:10.2f}, Std={orig_std:10.2f}")
        print(f"    Normalized: Mean={norm_mean:10.2f}, Std={norm_std:10.2f}")
    
    print("\nSkewed Features (Gap times in seconds):")
    for col in skewed_existing:
        orig_mean = df_original[col].mean()
        orig_std = df_original[col].std()
        print(f"  {col:40s}")
        print(f"    Original: Mean={orig_mean:10.2f}, Std={orig_std:10.2f}")
    
    print("\nSequential Features:")
    for col in sequential_existing:
        orig_min = df_original[col].min()
        orig_max = df_original[col].max()
        print(f"  {col:40s}")
        print(f"    Original range: [{orig_min:.0f}, {orig_max:.0f}]")
    
    # Plot original vs normalized distributions side-by-side
    print("\n" + "="*80)
    print("PLOTTING ORIGINAL vs NORMALIZED DISTRIBUTIONS")
    print("="*80)
    
    # 1. Gaussian features - Original vs Normalized
    if gaussian_existing:
        num_features = min(8, len(gaussian_existing))
        fig, axes = plt.subplots(num_features, 2, figsize=(16, 5*num_features))
        fig.suptitle('Gaussian Features: Original vs Normalized Distributions', fontsize=16, y=1.0)
        
        if num_features == 1:
            axes = axes.reshape(1, -1)
        
        for idx, col in enumerate(gaussian_existing[:num_features]):
            # Original distribution (left column)
            ax_orig = axes[idx, 0]
            data_orig = df_original[col].dropna()
            
            ax_orig.hist(data_orig, bins=50, alpha=0.7, edgecolor='black', density=True, color='steelblue')
            data_orig.plot.kde(ax=ax_orig, color='red', linewidth=2)
            
            mean_orig = data_orig.mean()
            std_orig = data_orig.std()
            skew_orig = stats.skew(data_orig)
            
            ax_orig.set_title(f'{col} (Original)\nMean={mean_orig:.2f}, Std={std_orig:.2f}, Skew={skew_orig:.2f}', fontsize=10)
            ax_orig.set_xlabel('Value (Original Scale)')
            ax_orig.set_ylabel('Density')
            ax_orig.axvline(mean_orig, color='green', linestyle='--', linewidth=2, label=f'Mean={mean_orig:.2f}')
            ax_orig.legend()
            
            # Normalized distribution (right column)
            ax_norm = axes[idx, 1]
            data_norm = df_normalized[col].dropna()
            
            ax_norm.hist(data_norm, bins=50, alpha=0.7, edgecolor='black', density=True, color='orange')
            data_norm.plot.kde(ax=ax_norm, color='red', linewidth=2)
            
            mean_norm = data_norm.mean()
            std_norm = data_norm.std()
            skew_norm = stats.skew(data_norm)
            
            ax_norm.set_title(f'{col} (Normalized)\nMean={mean_norm:.2f}, Std={std_norm:.2f}, Skew={skew_norm:.2f}', fontsize=10)
            ax_norm.set_xlabel('Value (Normalized Scale)')
            ax_norm.set_ylabel('Density')
            ax_norm.axvline(mean_norm, color='green', linestyle='--', linewidth=2, label=f'Mean={mean_norm:.2f}')
            ax_norm.axvline(0, color='black', linestyle=':', linewidth=1, alpha=0.5)
            ax_norm.legend()
        
        plt.tight_layout()
        display(plt.show())
    
    # 2. Skewed features - Original vs Normalized
    if skewed_existing:
        fig, axes = plt.subplots(len(skewed_existing), 2, figsize=(16, 5*len(skewed_existing)))
        fig.suptitle('Skewed Features: Original vs Normalized (after log1p + StandardScaler)', fontsize=16, y=1.0)
        
        if len(skewed_existing) == 1:
            axes = axes.reshape(1, -1)
        
        for idx, col in enumerate(skewed_existing):
            # Original distribution (left column)
            ax_orig = axes[idx, 0]
            data_orig = df_original[col].dropna()
            data_orig = data_orig[data_orig >= 0]  # Remove any negative gaps
            
            ax_orig.hist(data_orig, bins=50, alpha=0.7, edgecolor='black', color='blue')
            skew_orig = stats.skew(data_orig)
            mean_orig = data_orig.mean()
            median_orig = data_orig.median()
            
            ax_orig.set_title(f'{col} (Original)\nMean={mean_orig:.2f}s, Median={median_orig:.2f}s, Skew={skew_orig:.2f}', fontsize=10)
            ax_orig.set_xlabel('Value (seconds)')
            ax_orig.set_ylabel('Frequency')
            ax_orig.axvline(mean_orig, color='red', linestyle='--', linewidth=2, label=f'Mean={mean_orig:.2f}')
            ax_orig.axvline(median_orig, color='green', linestyle='--', linewidth=2, label=f'Median={median_orig:.2f}')
            ax_orig.legend()
            
            # Normalized distribution (right column)
            ax_norm = axes[idx, 1]
            data_norm = df_normalized[col].dropna()
            
            ax_norm.hist(data_norm, bins=50, alpha=0.7, edgecolor='black', color='orange')
            skew_norm = stats.skew(data_norm)
            mean_norm = data_norm.mean()
            median_norm = data_norm.median()
            
            ax_norm.set_title(f'{col} (Normalized)\nMean={mean_norm:.2f}, Median={median_norm:.2f}, Skew={skew_norm:.2f}', fontsize=10)
            ax_norm.set_xlabel('Value (normalized, after log1p)')
            ax_norm.set_ylabel('Frequency')
            ax_norm.axvline(mean_norm, color='red', linestyle='--', linewidth=2, label=f'Mean={mean_norm:.2f}')
            ax_norm.axvline(0, color='black', linestyle=':', linewidth=1, alpha=0.5, label='Zero')
            ax_norm.legend()
        
        plt.tight_layout()
        display(plt.show())
    
    # 3. Sequential features - Original vs Normalized
    if sequential_existing:
        fig, axes = plt.subplots(len(sequential_existing), 2, figsize=(16, 5*len(sequential_existing)))
        fig.suptitle('Sequential Features: Original vs Normalized (MinMaxScaler)', fontsize=16, y=1.0)
        
        if len(sequential_existing) == 1:
            axes = axes.reshape(1, -1)
        
        for idx, col in enumerate(sequential_existing):
            # Original distribution (left column)
            ax_orig = axes[idx, 0]
            data_orig = df_original[col].dropna()
            
            ax_orig.hist(data_orig, bins=50, alpha=0.7, edgecolor='black', color='purple')
            ax_orig.set_title(f'{col} (Original)\nRange=[{data_orig.min():.0f}, {data_orig.max():.0f}]', fontsize=10)
            ax_orig.set_xlabel('Value (Original Scale)')
            ax_orig.set_ylabel('Frequency')
            
            # Normalized distribution (right column)
            ax_norm = axes[idx, 1]
            data_norm = df_normalized[col].dropna()
            
            ax_norm.hist(data_norm, bins=50, alpha=0.7, edgecolor='black', color='orange')
            ax_norm.set_title(f'{col} (Normalized)\nRange=[{data_norm.min():.2f}, {data_norm.max():.2f}]', fontsize=10)
            ax_norm.set_xlabel('Value (Normalized [0,1] scale)')
            ax_norm.set_ylabel('Frequency')
            ax_norm.axhline(y=ax_norm.get_ylim()[1]*0.1, xmin=0, xmax=1, color='green', linestyle='--', linewidth=2, alpha=0.3)
        
        plt.tight_layout()
        display(plt.show())
        
else:
    print(f"⚠ Scaler file not found at {scaler_path}")
    print("The data in Unity Catalog is already normalized.")
    print("To see original distributions, the scalers would need to be saved during data creation.")
    print("\nCurrent normalized statistics are shown in Cell 5.")

In [0]:
#race completion % (lap number/circuit number of laps)
#check if tyrelife is a feature in the model
#total turning vs straight ratio
#mean vs std corner radius
#longest straight vs circuit length
#left right ratio vs number of turns
#ablations needed for driver/team embeddings